# Treinamento independente — detecção de armas e objetos perigosos

Treinamento de um detector YOLO leve com o **Dangerous Items Dataset**.

Classes do dataset: `machete`, `knife`, `baseball bat`, `rifle` e `gun`.

O dataset também contém imagens de fundo, usadas como exemplos negativos.

Fonte: Zenodo, DOI `10.5281/zenodo.16422779`.

In [ ]:
!pip -q install -U ultralytics pyyaml tqdm

In [ ]:
from dataclasses import dataclass, asdict
from pathlib import Path
from collections import Counter
import hashlib
import json
import random
import shutil
import subprocess
import sys
import zipfile

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import ultralytics
import yaml

from IPython.display import display, Image as ImagemNotebook
from tqdm.auto import tqdm
from ultralytics import YOLO


print("Ultralytics:", ultralytics.__version__)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 1. Configuração

In [ ]:
@dataclass(frozen=True)
class Configuracao:
    modelo_base: str = "yolo11n.pt"
    epocas: int = 80
    tamanho_imagem: int = 640
    tamanho_lote: int = 16
    paciencia: int = 15
    trabalhadores: int = 2
    semente: int = 42
    limiar_confianca: float = 0.25
    limiar_iou: float = 0.50
    fracao_calibracao_int8: float = 0.25


configuracao = Configuracao()

PASTA_RAIZ = Path("/content/deteccao-armas")
PASTA_DADOS = PASTA_RAIZ / "dataset"
PASTA_RESULTADOS = PASTA_RAIZ / "resultados"
ARQUIVO_ZIP = PASTA_RAIZ / "dangerous_items.zip"

URL_DATASET = (
    "https://zenodo.org/records/16422779/files/"
    "Dangerous%20Items.zip?download=1"
)
MD5_ESPERADO = "f567c3b6e070b2c36578a09c5c7eb9e2"

for pasta in (PASTA_RAIZ, PASTA_DADOS, PASTA_RESULTADOS):
    pasta.mkdir(parents=True, exist_ok=True)

random.seed(configuracao.semente)
np.random.seed(configuracao.semente)
torch.manual_seed(configuracao.semente)

display(pd.DataFrame([asdict(configuracao)]))

## 2. Baixar e extrair o dataset

In [ ]:
def calcular_md5(caminho: Path) -> str:
    resumo = hashlib.md5()
    with caminho.open("rb") as arquivo:
        for bloco in iter(lambda: arquivo.read(1024 * 1024), b""):
            resumo.update(bloco)
    return resumo.hexdigest()


if not ARQUIVO_ZIP.exists():
    subprocess.run(
        ["wget", "-c", URL_DATASET, "-O", str(ARQUIVO_ZIP)],
        check=True,
    )
else:
    print("ZIP já existente:", ARQUIVO_ZIP)

md5_obtido = calcular_md5(ARQUIVO_ZIP)
print("MD5 obtido:", md5_obtido)

if md5_obtido != MD5_ESPERADO:
    raise RuntimeError("Download corrompido. Apague o ZIP e baixe novamente.")

marcador = PASTA_DADOS / ".extraido"

if not marcador.exists():
    with zipfile.ZipFile(ARQUIVO_ZIP) as arquivo:
        for item in tqdm(
            arquivo.infolist(),
            desc="Extraindo",
            unit="arquivo",
        ):
            arquivo.extract(item, PASTA_DADOS)
    marcador.touch()
else:
    print("Dataset já extraído.")

## 3. Localizar as pastas de imagens


In [ ]:
EXTENSOES_IMAGEM = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp",
}


def possui_imagens(pasta: Path) -> bool:
    if not pasta.exists() or not pasta.is_dir():
        return False

    return any(
        caminho.is_file()
        and caminho.suffix.lower() in EXTENSOES_IMAGEM
        for caminho in pasta.iterdir()
    )


def encontrar_pasta_imagens(
    raiz: Path,
    nomes_split: tuple[str, ...],
) -> Path:
    """
    Procura imagens em estruturas como:

    raiz/train/images
    raiz/valid/images
    raiz/images/train
    raiz/images/valid
    raiz/train
    raiz/valid
    """

    candidatos = []

    for nome in nomes_split:
        candidatos.extend(
            [
                raiz / nome / "images",
                raiz / "images" / nome,
                raiz / nome,
            ]
        )

    for candidato in candidatos:
        if possui_imagens(candidato):
            return candidato.resolve()

    nomes_normalizados = {
        nome.lower()
        for nome in nomes_split
    }

    for pasta in raiz.rglob("*"):
        if not pasta.is_dir():
            continue

        if pasta.name.lower() not in nomes_normalizados:
            continue

        pasta_imagens = pasta / "images"

        if possui_imagens(pasta_imagens):
            return pasta_imagens.resolve()

        if possui_imagens(pasta):
            return pasta.resolve()

    raise FileNotFoundError(
        f"Não foi possível localizar o split: {nomes_split}"
    )


def carregar_nomes_classes(
    raiz: Path,
) -> dict[int, str]:
    """
    Procura um arquivo de nomes no dataset.

    Se não existir, usa a ordem informada pelo autor
    do Dangerous Items Dataset.
    """

    nomes_arquivos = {
        "classes.txt",
        "obj.names",
        "classes.names",
    }

    for caminho in raiz.rglob("*"):
        if (
            caminho.is_file()
            and caminho.name.lower() in nomes_arquivos
        ):
            nomes = [
                linha.strip()
                for linha in caminho.read_text(
                    encoding="utf-8",
                    errors="ignore",
                ).splitlines()
                if linha.strip()
            ]

            if len(nomes) == 5:
                print(
                    "Classes carregadas de:",
                    caminho,
                )

                return {
                    indice: nome
                    for indice, nome in enumerate(nomes)
                }

    print(
        "Arquivo de classes não encontrado. "
        "Usando o mapeamento conhecido do dataset."
    )

    return {
        0: "machete",
        1: "knife",
        2: "baseball_bat",
        3: "rifle",
        4: "gun",
    }


raiz_dataset = (
    PASTA_DADOS
    / "Dangerous Items"
)

if not raiz_dataset.exists():
    raiz_dataset = PASTA_DADOS

print(
    "Raiz do dataset:",
    raiz_dataset,
)

pasta_treino = encontrar_pasta_imagens(
    raiz_dataset,
    (
        "train",
        "training",
    ),
)

pasta_validacao = encontrar_pasta_imagens(
    raiz_dataset,
    (
        "valid",
        "val",
        "validation",
    ),
)

pasta_teste = encontrar_pasta_imagens(
    raiz_dataset,
    (
        "test",
        "testing",
    ),
)

classes_dataset = carregar_nomes_classes(
    raiz_dataset
)

dados_yaml = {
    "train": str(pasta_treino),
    "val": str(pasta_validacao),
    "test": str(pasta_teste),
    "nc": len(classes_dataset),
    "names": classes_dataset,
}

yaml_treinamento = (
    PASTA_RAIZ
    / "dados_armas.yaml"
)

yaml_treinamento.write_text(
    yaml.safe_dump(
        dados_yaml,
        allow_unicode=True,
        sort_keys=False,
    ),
    encoding="utf-8",
)

print("\nCaminhos encontrados:")
print("Treino    :", pasta_treino)
print("Validação :", pasta_validacao)
print("Teste     :", pasta_teste)

print("\nClasses:")
print(classes_dataset)

print("\nYAML criado:")
print(
    yaml_treinamento.read_text(
        encoding="utf-8"
    )
)

## 4. Auditar o dataset

Imagens sem rótulo ou com arquivo `.txt` vazio são contadas como negativas.

In [ ]:
EXTENSOES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

nomes = dados_yaml["names"]
classes = (
    {i: nome for i, nome in enumerate(nomes)}
    if isinstance(nomes, list)
    else {int(i): nome for i, nome in nomes.items()}
)


def listar_imagens(pasta: Path) -> list[Path]:
    return sorted(
        caminho
        for caminho in pasta.rglob("*")
        if caminho.suffix.lower() in EXTENSOES
    )


def caminho_rotulo(imagem: Path) -> Path:
    partes = list(imagem.parts)
    if "images" in partes:
        indice = len(partes) - 1 - partes[::-1].index("images")
        partes[indice] = "labels"
        return Path(*partes).with_suffix(".txt")
    return imagem.with_suffix(".txt")


def auditar(nome_split: str, pasta: Path):
    imagens = listar_imagens(pasta)
    objetos = Counter()
    negativas = 0
    invalidas = 0

    for imagem in tqdm(imagens, desc=f"Auditando {nome_split}", unit="imagem"):
        rotulo = caminho_rotulo(imagem)

        if not rotulo.exists():
            negativas += 1
            continue

        linhas = [l.strip() for l in rotulo.read_text().splitlines() if l.strip()]
        if not linhas:
            negativas += 1
            continue

        for linha in linhas:
            partes = linha.split()
            if len(partes) != 5:
                invalidas += 1
                continue
            try:
                classe = int(float(partes[0]))
                coordenadas = list(map(float, partes[1:]))
            except ValueError:
                invalidas += 1
                continue

            if classe not in classes or any(v < 0 or v > 1 for v in coordenadas):
                invalidas += 1
                continue

            objetos[classe] += 1

    resumo = {
        "split": nome_split,
        "imagens": len(imagens),
        "objetos": sum(objetos.values()),
        "negativas": negativas,
        "rotulos_invalidos": invalidas,
    }
    return resumo, objetos


resumos = []
contagens = []

for split in ("train", "val", "test"):
    if split not in dados_yaml:
        continue

    resumo, objetos = auditar(split, Path(dados_yaml[split]))
    resumos.append(resumo)

    for indice, nome in classes.items():
        contagens.append(
            {"split": split, "classe": nome, "objetos": objetos[indice]}
        )

tabela_resumo = pd.DataFrame(resumos)
tabela_classes = pd.DataFrame(contagens)

display(tabela_resumo)
display(
    tabela_classes.pivot(
        index="classe",
        columns="split",
        values="objetos",
    ).fillna(0).astype(int)
)

## 5. Visualizar exemplos

In [ ]:
imagens_treino = listar_imagens(Path(dados_yaml["train"]))
amostras = random.sample(imagens_treino, min(8, len(imagens_treino)))

plt.figure(figsize=(16, 8))

for indice, caminho in enumerate(amostras, start=1):
    imagem = cv2.cvtColor(cv2.imread(str(caminho)), cv2.COLOR_BGR2RGB)
    plt.subplot(2, 4, indice)
    plt.imshow(imagem)
    plt.title(caminho.name)
    plt.axis("off")

plt.tight_layout()
plt.show()

## 6. Treinar



In [ ]:
modelo = YOLO(configuracao.modelo_base)

resultado_treino = modelo.train(
    data=str(yaml_treinamento),
    epochs=configuracao.epocas,
    imgsz=configuracao.tamanho_imagem,
    batch=configuracao.tamanho_lote,
    patience=configuracao.paciencia,
    workers=configuracao.trabalhadores,
    device=0 if torch.cuda.is_available() else "cpu",
    project=str(PASTA_RESULTADOS),
    name="treinamento",
    exist_ok=True,
    pretrained=True,
    optimizer="auto",
    seed=configuracao.semente,
    deterministic=True,
    amp=True,
    cos_lr=True,
    close_mosaic=10,
    plots=True,
    save=True,
    save_period=10,
    verbose=True,
)

## 7. Histórico e gráficos

In [ ]:
PASTA_TREINO = PASTA_RESULTADOS / "treinamento"
MELHOR_MODELO = PASTA_TREINO / "weights" / "best.pt"
ULTIMO_MODELO = PASTA_TREINO / "weights" / "last.pt"

historico = pd.read_csv(PASTA_TREINO / "results.csv")
historico.columns = [coluna.strip() for coluna in historico.columns]
display(historico)


def plotar(colunas: list[str], titulo: str, eixo_y: str):
    existentes = [coluna for coluna in colunas if coluna in historico.columns]

    plt.figure(figsize=(10, 6))
    for coluna in existentes:
        plt.plot(historico["epoch"], historico[coluna], label=coluna)

    plt.xlabel("Época")
    plt.ylabel(eixo_y)
    plt.title(titulo)
    plt.legend()
    plt.grid()
    plt.show()


plotar(["train/box_loss", "val/box_loss"], "Perda das caixas", "Box loss")
plotar(["train/cls_loss", "val/cls_loss"], "Perda de classificação", "Cls loss")
plotar(
    ["metrics/precision(B)", "metrics/recall(B)"],
    "Precisão e revocação",
    "Métrica",
)
plotar(
    ["metrics/mAP50(B)", "metrics/mAP50-95(B)"],
    "mAP",
    "mAP",
)

In [ ]:
for nome in [
    "results.png",
    "confusion_matrix.png",
    "confusion_matrix_normalized.png",
    "PR_curve.png",
    "F1_curve.png",
    "P_curve.png",
    "R_curve.png",
    "labels.jpg",
]:
    caminho = PASTA_TREINO / nome
    if caminho.exists():
        print(nome)
        display(ImagemNotebook(filename=str(caminho)))

## 8. Avaliar no conjunto de teste

In [ ]:
if "test" not in dados_yaml:
    raise RuntimeError("O dataset não possui conjunto de teste.")

melhor_modelo = YOLO(str(MELHOR_MODELO))

metricas_teste = melhor_modelo.val(
    data=str(yaml_treinamento),
    split="test",
    imgsz=configuracao.tamanho_imagem,
    batch=configuracao.tamanho_lote,
    conf=configuracao.limiar_confianca,
    iou=configuracao.limiar_iou,
    device=0 if torch.cuda.is_available() else "cpu",
    project=str(PASTA_RESULTADOS),
    name="teste",
    exist_ok=True,
    plots=True,
    verbose=True,
)

resumo_teste = pd.DataFrame([{
    "mAP50-95": float(metricas_teste.box.map),
    "mAP50": float(metricas_teste.box.map50),
    "mAP75": float(metricas_teste.box.map75),
    "precisao_media": float(np.mean(metricas_teste.box.p)),
    "revocacao_media": float(np.mean(metricas_teste.box.r)),
}])

metricas_por_classe = pd.DataFrame({
    "classe": [
        metricas_teste.names[i]
        for i in range(len(metricas_teste.box.maps))
    ],
    "precisao": metricas_teste.box.p,
    "revocacao": metricas_teste.box.r,
    "mAP50": metricas_teste.box.ap50,
    "mAP50-95": metricas_teste.box.maps,
})

display(resumo_teste)
display(metricas_por_classe.sort_values("mAP50-95", ascending=False))

## 9. Predições de exemplo

In [ ]:
imagens_teste = listar_imagens(Path(dados_yaml["test"]))
amostras_teste = random.sample(imagens_teste, min(8, len(imagens_teste)))

melhor_modelo.predict(
    source=[str(caminho) for caminho in amostras_teste],
    imgsz=configuracao.tamanho_imagem,
    conf=configuracao.limiar_confianca,
    iou=configuracao.limiar_iou,
    device=0 if torch.cuda.is_available() else "cpu",
    project=str(PASTA_RESULTADOS),
    name="predicoes",
    exist_ok=True,
    save=True,
    verbose=True,
)

for caminho in sorted((PASTA_RESULTADOS / "predicoes").glob("*")):
    if caminho.suffix.lower() in EXTENSOES:
        display(ImagemNotebook(filename=str(caminho)))

## 10. Exportar para TensorFlow Lite

A primeira célula exporta Float32. A segunda exporta INT8 para uso posterior na Raspberry Pi.

In [ ]:
tflite_float32 = melhor_modelo.export(
    format="tflite",
    imgsz=configuracao.tamanho_imagem,
    int8=False,
    half=False,
    nms=True,
    batch=1,
)

print("TFLite Float32:", tflite_float32)

In [ ]:
tflite_int8 = melhor_modelo.export(
    format="tflite",
    imgsz=configuracao.tamanho_imagem,
    int8=True,
    data=str(yaml_treinamento),
    fraction=configuracao.fracao_calibracao_int8,
    nms=True,
    batch=1,
)

print("TFLite INT8:", tflite_int8)

## 11. Salvar e baixar os resultados

In [ ]:
from google.colab import files

resumo = {
    "configuracao": asdict(configuracao),
    "versoes": {
        "ultralytics": ultralytics.__version__,
        "torch": torch.__version__,
    },
    "classes": classes,
    "auditoria": tabela_resumo.to_dict(orient="records"),
    "teste": resumo_teste.iloc[0].to_dict(),
}

(PASTA_RESULTADOS / "resumo_experimento.json").write_text(
    json.dumps(resumo, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

metricas_por_classe.to_csv(
    PASTA_RESULTADOS / "metricas_teste_por_classe.csv",
    index=False,
)
tabela_resumo.to_csv(
    PASTA_RESULTADOS / "auditoria_dataset.csv",
    index=False,
)

PASTA_ENTREGA = PASTA_RAIZ / "entrega"
if PASTA_ENTREGA.exists():
    shutil.rmtree(PASTA_ENTREGA)
PASTA_ENTREGA.mkdir()

for origem in [
    PASTA_TREINO,
    PASTA_RESULTADOS / "teste",
    PASTA_RESULTADOS / "predicoes",
    PASTA_RESULTADOS / "resumo_experimento.json",
    PASTA_RESULTADOS / "metricas_teste_por_classe.csv",
    PASTA_RESULTADOS / "auditoria_dataset.csv",
    yaml_treinamento,
]:
    origem = Path(origem)
    if not origem.exists():
        continue

    destino = PASTA_ENTREGA / origem.name
    if origem.is_dir():
        shutil.copytree(origem, destino, dirs_exist_ok=True)
    else:
        shutil.copy2(origem, destino)

for exportado in [Path(tflite_float32), Path(tflite_int8)]:
    if exportado.exists():
        destino = PASTA_ENTREGA / exportado.name
        if exportado.is_dir():
            shutil.copytree(exportado, destino, dirs_exist_ok=True)
        else:
            shutil.copy2(exportado, destino)

arquivo_final = shutil.make_archive(
    "/content/resultados_deteccao_armas",
    "zip",
    root_dir=str(PASTA_ENTREGA),
)

print("Arquivo:", arquivo_final)
files.download(arquivo_final)